# Module 12: Regression Discontinuity

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

When a threshold on a continuous variable decides who gets a program, units
just above and just below it are comparable, and the jump in the outcome at
the threshold identifies the effect **without any parallel trends assumption.**

The selection rule in this dataset is **close** to a threshold. This module is
about why close is not enough, and about the two separate things that have to
be true.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

In [ ]:
X = profile.set_index("agency_id").copy()
X = X.loc[sorted(X.index)]
X["treated"] = [1 if a in TRAINED else 0 for a in X.index]
X["lpop"] = np.log(X["population_served"])
X["lsworn"] = np.log(X["sworn_officers"])
print(f"{len(X)} agencies, {X['treated'].sum()} of them treated")

## 2. The design and its assumptions

> assignment is determined by whether a **running variable** crosses a cutoff
>
> the effect is the jump in the outcome at the cutoff

| Assumption | Diagnostic |
|---|---|
| **Sharpness** | does the cutoff perfectly predict treatment |
| **Continuity** | would the outcome have been smooth at the cutoff without the program |
| **No manipulation** | is the density of the running variable smooth at the cutoff |
| **Enough mass near the cutoff** | count the units inside a defensible bandwidth |

The first and last are arithmetic. The middle two are arguments.

## 3. How sharp is the cutoff

In [ ]:
v = X["pre_program_uof_per_100_arrests"].sort_values(ascending=False)
CUT = 3.159
for a, x in v.items():
    side = "above" if x > CUT else "below"
    tr = a in TRAINED
    wrong = tr != (x > CUT)
    flag = "   <- on the wrong side" if wrong else ""
    print(f"  {x:.3f}  {side:5s}  {'treated' if tr else 'control':7s}  "
          f"{NAME[a]:34s}{flag}")
ab = [a for a in v.index if v[a] > CUT]
be = [a for a in v.index if v[a] <= CUT]
print(f"\n  above the cutoff: {sum(a in TRAINED for a in ab)} of {len(ab)} treated")
print(f"  below the cutoff: {sum(a in TRAINED for a in be)} of {len(be)} treated")

**Two of twelve fall on the wrong side**, one in each direction. The design is
therefore **fuzzy**, not sharp, which is not fatal by itself: a fuzzy design
uses the cutoff as an instrument for treatment.

But a fuzzy design inherits every requirement from
[Module 11](Module_11_Instrumental_Variables.ipynb), including a first stage
strong enough to divide by.

In [ ]:
X["above"] = (X["pre_program_uof_per_100_arrests"] > CUT).astype(int)
fs = sm.OLS(X["treated"], sm.add_constant(X[["above"]])).fit()
print(f"  the cutoff as an instrument for treatment")
print(f"    first stage coefficient {fs.params['above']:+.3f}, "
      f"F = {fs.fvalue:.2f}")
print(f"    compare the usual bar of 10")

The first stage F is **7.60**, below the conventional bar of 10 and not
negligible. Fuzziness is a real cost here and it is not the binding problem,
which is the next section.

## 4. The problem that is binding

In [ ]:
for bw in [0.15, 0.30, 0.50, 0.80, 1.20]:
    inside = X[np.abs(X["pre_program_uof_per_100_arrests"] - CUT) < bw]
    print(f"  bandwidth {bw:.2f}   {len(inside):2d} agencies   "
          f"{inside['treated'].sum()} treated, {len(inside) - inside['treated'].sum()} control")

Regression discontinuity estimates a **local** effect, at the cutoff. Here the
local sample is **two agencies**: Summit County just below and Havenbrook just
above.

Widening the bandwidth to include more units means the estimate is no longer
local, and at a bandwidth of 0.80 it contains eight of the twelve agencies,
which is most of the panel. **There is no bandwidth that is both local and
populated.**

And the two agencies at the cutoff are the two least suitable in the dataset:
Summit County is the pre trend violator excluded from the main analysis, and
Havenbrook is the agency that reclassified its call categories in January
2023.

## 5. The other assumption, which has no data here

Continuity says the outcome would have been smooth across the cutoff without
the program. With two units near the cutoff there is nothing to fit a local
polynomial to, so the assumption cannot be examined, let alone tested.

In [ ]:
import warnings as _w
_w.filterwarnings("ignore")

agg = f[f["period"] == "after"].groupby("agency_id").apply(
    lambda g: 100 * g["n_uof"].sum() / g["n_arrests"].sum())
X["post_rate"] = agg
for bw in [0.50, 0.80, 1.20]:
    inside = X[np.abs(X["pre_program_uof_per_100_arrests"] - CUT) < bw]
    z = sm.OLS(inside["post_rate"],
               sm.add_constant(inside[["above",
                                       "pre_program_uof_per_100_arrests"]])).fit()
    print(f"  bandwidth {bw:.2f}, {len(inside)} agencies:  "
          f"jump at the cutoff {z.params['above']:+.3f} rate points   "
          f"p = {z.pvalues['above']:.3f}")
print(f"\n  the program's true effect is a {abs(TRUTH):.0f} percent reduction, "
      f"which on a base near 3.2 is about 0.38 rate points")

Three bandwidths, and all three report a jump of about **plus one rate
point**, with p values of 0.02, 0.01 and 0.008.

**The true jump is about minus 0.38 points.** The design returns the wrong
sign, at every bandwidth, significantly.

The reason is visible in section 4: with six to twelve points, a linear
control for the running variable cannot absorb the fact that agencies above
the cutoff have structurally higher rates, so the level difference that the
selection rule created is read as a jump.

**This is what a regression discontinuity produces when it should not have
been run**: numbers with p values attached, pointing the wrong way. The
diagnostic that should have stopped it is the count of units near the cutoff,
and it required no fitting.

## 6. When regression discontinuity is the right tool

| Requirement | Here |
|---|---|
| A rule stated in advance with a numeric cutoff | approximately |
| Many units near the cutoff | **two** |
| A running variable measured before assignment | yes |
| No ability to manipulate position | plausible, agencies did not choose their rate |
| Continuity of everything else at the cutoff | untestable with two units |

**Public safety programs often have something that looks like a threshold and
almost never have enough agencies near it.** A state with three hundred
agencies and a rule applied to all of them is a different situation, and there
the design becomes worth considering.

## Exercise

The cutoff was chosen by hand at 3.159. Find out how much the answer depends
on that choice.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for cut in [2.90, 3.05, 3.159, 3.30, 3.45]:
        X["ab"] = (X["pre_program_uof_per_100_arrests"] > cut).astype(int)
        wrong = int((X["ab"] != X["treated"]).sum())
        inside = X[np.abs(X["pre_program_uof_per_100_arrests"] - cut) < 0.80]
        z = sm.OLS(inside["post_rate"],
                   sm.add_constant(inside[["ab",
                                           "pre_program_uof_per_100_arrests"]])).fit()
        rows.append({"cutoff": cut, "agencies on the wrong side": wrong,
                     "agencies within 0.80": len(inside),
                     "estimated jump": f"{z.params['ab']:+.3f}"})
    print("  the true jump is about -0.38 rate points\n")
    display(pd.DataFrame(rows).set_index("cutoff"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The estimated jump moves substantially across cutoffs that are all defensible
on the face of it, and the number of misclassified agencies changes too.

**When the cutoff itself is a judgment call, the design has lost the property
that makes it attractive.** The appeal of regression discontinuity is that
assignment near the threshold is as good as random because nobody controls
which side of a published line they land on. That argument requires a
published line.

Here the rule was "the five agencies with the highest rates", which is a rank
based rule rather than a threshold rule. **Rank based selection does not
create a discontinuity**, because the cutoff depends on the whole
distribution rather than on a fixed value, and it moves whenever the set of
candidate agencies changes.

The distinction is worth carrying into any conversation with a program
administrator: "the worst five" and "everyone above 3.0" look similar and only
the second one supports this design.

</details>

---

**Next:** [Module 13: Placebo, Permutation and Falsification Tests](Module_13_Placebo_Permutation_And_Falsification.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*